# CNN Image Classification — CIFAR-10
## Neural networks for energy applications — Day 4, Exercise 1

In this exercise we build and train convolutional neural networks (CNNs) for image
classification on the CIFAR-10 dataset. We start with a simple two-block architecture,
evaluate its performance, diagnose errors, and then change one thing, adding batch
normalization and dropout, to see whether it helps.

**Evaluation protocol.** We hold out 5,000 of the 50,000 training images as a
**validation set**. Validation data are used to monitor training, to keep the best epoch,
to inspect errors and to compare the two models. The 10,000 **test** images are used once,
in Section 8, for the model we have already chosen. Using the test set after every epoch
(or to pick the better model) would turn it into a second validation set.

**Runtime.** To fit a laptop CPU, both models train on the same fixed 20,000-image subset
of the training split for 10 epochs.
This takes about 3–5 minutes on a laptop CPU and 1–2 minutes on a GPU or Apple silicon.
Set `N_TRAIN = 45_000` to use the whole training split; the comparison stays fair as long
as both models get the same budget.

---
## 0 — Imports and Setup

In [ ]:
import copy
import os

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split

import torchvision
import torchvision.transforms as transforms

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Use a GPU if there is one (CUDA, or MPS on Apple silicon), otherwise the CPU.
# GPU results can differ slightly from CPU results even with the same seed.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# CIFAR-10 is downloaded to ./data unless the COURSE_DATA environment variable names another folder
DATA_ROOT = os.environ.get("COURSE_DATA", "./data")
print(f"Data folder: {DATA_ROOT}")
print("Setup complete.")

---
## 1 — Load CIFAR-10 Dataset

CIFAR-10 consists of 60,000 colour images (32x32 pixels, 3 channels) across 10 classes:
airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

### 1.1 Define Transforms, Download Data and Hold Out a Validation Set

The images are independent photographs, so a random validation split is acceptable here
(for patches cut from the same well or image it would not be: see Lab 2B).

In [ ]:
# YOUR CODE HERE
# TODO: Define transform_basic using transforms.Compose with:
#       - transforms.ToTensor()
#       - transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
#         (per-channel mean and std of the CIFAR-10 training images)
# TODO: Download the CIFAR-10 training and test datasets using torchvision.datasets.CIFAR10
#       with root=DATA_ROOT, download=True, transform=transform_basic
# TODO: Define the class names tuple in label order and check it against full_train_dataset.classes
# TODO: Hold out 5,000 training images for validation:
#       train_set, val_set = random_split(full_train_dataset, [45000, 5000],
#                                         generator=torch.Generator().manual_seed(RANDOM_STATE))
# TODO: Classroom budget: train_subset = Subset(train_set, <first N_TRAIN = 20_000 indices of a
#       seeded torch.randperm(len(train_set))>)
# TODO: Print the number of training/validation/test samples, the image shape and the label mapping

### 1.2 Visualize Sample Images

In [ ]:
# YOUR CODE HERE
# TODO: Write a helper function imshow(img) that unnormalizes an image tensor and returns
#       an H x W x C array for plt.imshow
#       Hint: reverse the normalization by multiplying by std and adding mean
# TODO: Create a 2x8 subplot figure showing 16 training images with class labels

### 1.3 Create DataLoaders

In [ ]:
# YOUR CODE HERE
# TODO: Create train_loader (from train_subset, shuffle=True) and val_loader (from val_set,
#       shuffle=False) using DataLoader with BATCH_SIZE=64 and num_workers=0
#       (no test loader yet: the test set is used once, in Section 8)
# TODO: Verify the batch shape by fetching one batch with next(iter(train_loader))

---
## 2 — Build a Simple CNN

Our **SimpleCNN** has two convolutional blocks followed by two fully connected layers:

| Block | Layers |
|-------|--------|
| Conv Block 1 | Conv2d(3, 32, 3, padding=1) + ReLU + MaxPool2d(2) |
| Conv Block 2 | Conv2d(32, 64, 3, padding=1) + ReLU + MaxPool2d(2) |
| Classifier | Flatten + Linear(64*8*8, 512) + ReLU + Linear(512, 10) |

Before training we pass a dummy batch through the model and print the output shape of every
layer: the shape ledger from the slides.

In [ ]:
# YOUR CODE HERE
# TODO: Define class SimpleCNN(nn.Module) with:
#   __init__: conv1 (3->32, kernel=3, pad=1), conv2 (32->64, kernel=3, pad=1),
#             pool (MaxPool2d(2,2)), fc1 (64*8*8 -> 512), fc2 (512 -> 10)
#   forward:  pool(relu(conv1(x))) -> pool(relu(conv2(x))) -> flatten -> relu(fc1) -> fc2
# Hint: After two MaxPool2d(2) layers, 32x32 becomes 8x8
# TODO: Instantiate the model on device and print it
# TODO: Count and print total parameters
# TODO: Shape check: register a forward hook on every Conv2d, MaxPool2d and Linear layer
#       (model.named_children()) that prints the layer name and output shape, pass a dummy
#       batch torch.zeros(8, 3, 32, 32) through the model in eval mode under torch.no_grad(),
#       then remove the hooks


---
## 3 — Training Loop

### 3.1 Define Loss and Optimizer

In [ ]:
# YOUR CODE HERE
# TODO: Define criterion = nn.CrossEntropyLoss()
# TODO: Define optimizer = optim.Adam(model.parameters(), lr=1e-3)


### 3.2 Training and Validation Functions

In [ ]:
# YOUR CODE HERE
# TODO: Define train_one_epoch(model, loader, criterion, optimizer, device):
#       - Set model.train()
#       - Loop over batches: zero_grad, forward, loss, backward, step
#       - Track running_loss, correct, total
#       - Return (avg_loss, accuracy)
#
# TODO: Define evaluate(model, loader, criterion, device):
#       - Set model.eval() and use torch.no_grad()
#       - Loop over batches: forward, compute loss
#       - Track running_loss, correct, total
#       - Return (avg_loss, accuracy)


### 3.3 Train for 10 Epochs

After every epoch we evaluate on the **validation** set and keep a copy of the weights from
the epoch with the highest validation accuracy.

In [ ]:
# YOUR CODE HERE
# TODO: Set NUM_EPOCHS = 10 (classroom budget; with more time, raise it together with N_TRAIN)
# TODO: Create a history dict with keys: train_loss, train_acc, val_loss, val_acc
# TODO: Loop over epochs, calling train_one_epoch on train_loader and evaluate on val_loader
#       (NOT on the test set)
# TODO: Append results to history and print epoch stats
# TODO: Keep best_state = copy.deepcopy(model.state_dict()) whenever the validation accuracy
#       improves, and reload it after training with model.load_state_dict(best_state)

---
## 4 — Plot Training Curves

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot figure
#       - Left: training and validation loss vs epoch
#       - Right: training and validation accuracy vs epoch, with the kept epoch marked
# TODO: Add labels, legends, and a title

---
## 5 — Evaluate on the Validation Set

Error analysis is part of model development, so it uses validation images. The test set is
kept for one final check (Section 8).

### 5.1 Overall Validation Accuracy

In [ ]:
# YOUR CODE HERE
# TODO: Call evaluate() on the validation set and print the validation loss and accuracy

### 5.2 Per-Class Accuracy (validation)

In [ ]:
# YOUR CODE HERE
# TODO: Loop over the validation set and count correct predictions per class
# Hint: use two lists class_correct and class_total, each of length 10
# TODO: Print a table of class name, correct count, total count, accuracy

---
## 6 — Visualize Misclassified Validation Images

In [ ]:
# YOUR CODE HERE
# TODO: Collect 8 misclassified examples from the validation set
#       Store tuples of (image, true_label, predicted_label)
# TODO: Display them in a 2x4 grid with title showing true vs predicted class
# Hint: compare predicted != labels to find misclassified samples

---
## 7 — Improved CNN with BatchNorm and Dropout

We change **one thing**: we add **Batch Normalization** after each convolution (stabilizes
training) and **Dropout** before the final layer (reduces overfitting of the large dense
head). Everything else stays as in SimpleCNN: layers and sizes, optimizer, training images,
epoch budget and seed. A difference on the validation set can then be attributed to this
change.

### 7.1 Define Improved Architecture

In [ ]:
# YOUR CODE HERE
# TODO: Define class ImprovedCNN(nn.Module): SimpleCNN with ONE change, BatchNorm and Dropout:
#   Block 1: Conv2d(3, 32, 3, pad=1) + BatchNorm2d(32) + ReLU + MaxPool
#   Block 2: Conv2d(32, 64, 3, pad=1) + BatchNorm2d(64) + ReLU + MaxPool
#   Classifier: Flatten + Linear(64*8*8, 512) + ReLU + Dropout(0.5) + Linear(512, 10)
# Hint: after 2 pooling layers, 32 -> 16 -> 8 (as in SimpleCNN)
# TODO: Call torch.manual_seed(RANDOM_STATE), instantiate on device and print parameter count

### 7.2 Train the Improved Model

In [ ]:
# YOUR CODE HERE
# TODO: Define loss, optimizer (Adam, lr=1e-3) for the improved model
# TODO: Train for the SAME NUM_EPOCHS on the same train_loader, evaluating on val_loader;
#       store results in history_v2
# TODO: Keep the best validation weights (best_state_v2) and reload them after training
# TODO: Print epoch stats each iteration

### 7.3 Compare Validation Curves and Choose a Model

We choose the model with the higher best validation accuracy. Differences of a few tenths of
a percent are within run-to-run noise for a single seed; in that case prefer the simpler model.

In [ ]:
# YOUR CODE HERE
# TODO: Create a 1x2 subplot comparing SimpleCNN vs ImprovedCNN:
#       - Left: validation accuracy curves for both models
#       - Right: validation loss curves for both models
# TODO: Print the best validation accuracy for each model
# TODO: Choose final_model and final_name: the model with the higher best validation accuracy

---
## 8 — Final Test Evaluation (once)

The choice is now locked: the model with the higher validation accuracy, with the weights of
its best validation epoch. We evaluate it once on the 10,000 test images. This number
describes the locked procedure; changing the model after seeing it would turn the test set
into another validation set.

In [ ]:
# YOUR CODE HERE
# TODO: Create test_loader (shuffle=False, num_workers=0)
# TODO: Call evaluate() ONCE on the test set with final_model and print the final test
#       loss and accuracy

---
## Summary

In this exercise we:

1. **Loaded and explored** the CIFAR-10 dataset (32x32 colour images, 10 classes) and held out
   5,000 training images for validation.
2. **Built a SimpleCNN** with two convolutional blocks and two fully connected layers, and
   printed every layer's output shape for a dummy batch.
3. **Trained** the model and tracked training/validation loss and accuracy, keeping the best
   validation epoch.
4. **Evaluated** per-class accuracy and visualized misclassified images on the validation set.
5. **Changed one thing**, adding BatchNorm and Dropout, and trained with the same budget.
6. **Compared** the two models on validation data, chose one, and evaluated it once on the
   test set.

Key takeaways:
- Model and epoch selection use validation data; the test set is touched once, at the end.
- Change one thing per comparison and give both models the same budget.
- Batch normalization often speeds up convergence, and dropout can reduce overfitting of a
  large dense head; check both claims on validation data rather than assuming them.